https://arxiv.org/pdf/2602.22801

## 1. 动机：为什么需要 Hybrid Loss？

在端到端自动驾驶的轨迹预测中，模型需要输出未来一段时间内的运动轨迹。通常有两种表示方式：

- **Waypoint representation**：直接预测每个未来时刻的绝对坐标 $(x_t, y_t)$。  
  优点：能很好地捕捉轨迹的全局几何形状，ADE（平均位移误差）较低。  
  缺点：生成的速度曲线容易出现锯齿状抖动（jerk），局部时序平滑性差，影响舒适性和闭环稳定性。

- **Velocity representation**：预测相邻时刻的位移增量 $(\Delta x_t, \Delta y_t)$，再通过积分恢复绝对位置。  
  优点：速度曲线平滑，数值分布更集中，训练更稳定，闭环舒适度高。  
  缺点：积分过程会累积误差，可能导致最终位置偏离真实轨迹，全局几何精度下降。

论文实验（Figure 5）表明：  
- 纯 waypoint 监督：ADE 好，但速度抖动严重，舒适度低。  
- 纯 velocity 监督：速度平滑，舒适度高，但 ADE 稍差。

因此，一个自然的想法是**同时监督两种表示**，让模型既保持局部平滑，又兼顾全局几何精度。这就是 **Hybrid Loss** 的动机。

---

## 2. Hybrid Loss 的数学形式

设模型输出的速度/位移增量为 $\tau_\theta^v$，真实速度为 $\tau_0^v$，积分矩阵 $M$ 为下三角全 1 矩阵（将速度累加为位置）。  
定义两项损失：

$$
\mathcal{L}_{\text{velocity}} = \mathbb{E}\left[ \|\tau_\theta^v - \tau_0^v\|_2^2 \right]
$$

$$
\mathcal{L}_{\text{waypoint}} = \mathbb{E}\left[ \|M(\tau_\theta^v \Delta t) - M(\tau_0^v \Delta t)\|_2^2 \right]
$$

混合损失为：

$$
\mathcal{L}_{\text{hybrid}} = \mathcal{L}_{\text{velocity}} + \omega \cdot \mathcal{L}_{\text{waypoint}}
$$

其中 $\omega$ 是平衡权重。

将两项合并，可以写成统一的二次型：

$$
\mathcal{L}_{\text{hybrid}} = \mathbb{E}\left[ (\tau_\theta^v - \tau_0^v)^\top P (\tau_\theta^v - \tau_0^v) \right]
$$

其中

$$
P = I + \Delta t^2 \omega M^\top M
$$

这就是论文中的 **Theorem 4.2** 的核心形式。

---

## 3. 为什么不破坏理论等价性？

关键在于 $P$ 是**正定矩阵**（positive definite）。

对任意非零向量 $z$：

$$
z^\top P z = z^\top z + \Delta t^2 \omega z^\top M^\top M z
= \|z\|_2^2 + \Delta t^2 \omega \|M z\|_2^2 > 0
$$

因为 $\omega \ge 0$，且 $\|z\|_2^2 > 0$，所以 $P \succ 0$。

### 3.1 Bregman 散度性质

论文引用了 **Lemma 4.1**（Lipman et al., 2024）：  
> 任何正定矩阵 $P$ 定义的二次型 $D_P(u,v) = (u-v)^\top P (u-v)$ 都是 Bregman 散度。

Bregman 散度有一个重要性质：对于条件回归问题

$$
\min_f \mathbb{E}\left[ D_P(f(x), y) \mid x \right]
$$

其唯一最优解是条件期望：

$$
f^*(x) = \mathbb{E}[y \mid x]
$$

### 3.2 对扩散模型的意义

在扩散模型中，训练目标是让模型在给定带噪输入 $\tau_t^v$ 时，预测出能够恢复正确 score function 的量。无论参数化选择是预测 $\tau_0$、噪声 $\epsilon$ 还是 flow velocity $v_t$，它们之间可以通过线性变换互相转换。

如果用 $\tau_0$-prediction 和 $\tau_0$-loss，普通 MSE 损失的最优解就是 $\mathbb{E}[\tau_0^v \mid \tau_t^v]$。根据 Tweedie 公式，这个条件期望恰好能恢复 marginal score function：

$$
\nabla_{\tau_t^v} \log p_t(\tau_t^v) = \frac{\alpha_t \mathbb{E}[\tau_0^v \mid \tau_t^v] - \tau_t^v}{\sigma_t^2}
$$

因此，只要损失函数的最优解仍然是条件期望，就**不会改变扩散模型的理论学习目标**。

对于 hybrid loss，由于 $P \succ 0$，它是一个 Bregman 散度，其最小化器仍然是 $\mathbb{E}[\tau_0^v \mid \tau_t^v]$。所以：

$$
\boxed{
\text{Hybrid loss 在理论上与标准 MSE 损失等价，都能恢复正确的 marginal score function。}
}
$$

这就是 Theorem 4.2 的核心结论：**Hybrid loss 只是改变了误差不同方向上的加权（通过 $P$ 矩阵），但不会引入偏差**。

---

## 4. 归一化和截断梯度为什么不破坏等价性？

### 4.1 状态归一化

实际代码中，输入输出都经过归一化处理。归一化相当于对误差进行线性缩放：

$$
E_{\text{normalized}} = (A_\theta - A_0) S^{-1}
$$

其中 $S = \mathrm{diag}(\sigma_x, \sigma_y, \sigma_{\cos}, \sigma_{\sin})$。

在归一化空间下，waypoint loss 中的 $P$ 变为：

$$
P_d = I + \omega \sigma_d^2 M^\top M
$$

只要 $\sigma_d > 0$，$P_d$ 仍然是正定的（因为 $\sigma_d^2 > 0$ 不改变正定性）。因此归一化**不改变损失函数的 Bregman 散度性质**，只是重新调整了各维度的权重，使其在数值上更稳定。

### 4.2 截断梯度（Detached Integral）

论文中提到，由于积分矩阵 $M$ 的存在，waypoint loss 对较早时刻的速度增量会产生很大的梯度累积，导致训练不稳定。为此，他们提出**截断梯度**：

- 前向计算时，仍然使用完整的积分，因此损失函数的数值与标准 P 范数完全一致。
- 反向传播时，只允许梯度回传到最近 $W$ 个速度增量，更早的历史通过 stop-gradient 截断。

数学上，这相当于使用了一个**代理梯度**，但前向目标函数不变：

$$
\mathcal{L}_{\text{forward}} = \frac{1}{T} e^\top P e \quad \text{（精确）}
$$

$$
\nabla_{\text{backward}} \approx B_W^\top M e \quad \text{（截断代理）}
$$

因为**前向损失没有改变**，其理论最小值仍然对应条件期望，所以截断梯度**不改变理论最优解**，只是训练过程中的一种工程稳定化手段。论文明确指出：

> This preserves the forward value of Lhybrid and does not alter the minimizer characterized by Theorem 4.2; it serves purely as training stabilization.

---

## 5. 总结

- **动机**：结合 waypoint 的全局几何精度与 velocity 的局部时序平滑。
- **理论保证**：混合损失等价于一个正定 $P$ 范数，是 Bregman 散度，其最小化器与普通 MSE 相同，因此不破坏扩散模型 score matching 的理论正确性。
- **工程实践**：
  - 归一化调整各维度尺度，但不改变正定性。
  - 截断梯度仅改变反向传播路径，前向损失保持不变，因此不影响理论最优解。

这种设计使得 Hybrid Loss 在提升闭环性能的同时，保持了扩散模型学习目标的无偏性，是理论与实践结合的典型范例。

# Hybrid Loss 二次型详细推导

## 1. 轨迹表示与动作转换

设预测时域长度为 $T$。单个样本的真实绝对轨迹为：

$$
X_0 =
\begin{bmatrix}
x_1 & y_1 & \cos\theta_1 & \sin\theta_1 \\
x_2 & y_2 & \cos\theta_2 & \sin\theta_2 \\
\vdots & \vdots & \vdots & \vdots \\
x_T & y_T & \cos\theta_T & \sin\theta_T
\end{bmatrix} \in \mathbb{R}^{T \times 4}.
$$

我们不直接预测绝对位置，而是将其转换为相邻帧的位移增量。定义：

$$
\Delta x_t = x_t - x_{t-1}, \qquad \Delta y_t = y_t - y_{t-1}, \quad t=1,\dots,T,
$$

并规定 $x_0 = y_0 = 0$。航向角的正余弦不做差分，保持不变。于是得到模型的动作目标：

$$
A_0 =
\begin{bmatrix}
\Delta x_1 & \Delta y_1 & \cos\theta_1 & \sin\theta_1 \\
\Delta x_2 & \Delta y_2 & \cos\theta_2 & \sin\theta_2 \\
\vdots & \vdots & \vdots & \vdots \\
\Delta x_T & \Delta y_T & \cos\theta_T & \sin\theta_T
\end{bmatrix} \in \mathbb{R}^{T \times 4}.
$$

> **注**：这里的 $\Delta x,\Delta y$ 已经包含了相邻帧之间的时间间隔，因此代码中将其视为位移增量（displacement increment），不再额外乘以 $\Delta t$。

## 2. 归一化空间

为了数值稳定，对动作的每个维度进行仿射归一化。设均值向量 $\mu \in \mathbb{R}^4$，标准差对角矩阵

$$
S = \operatorname{diag}(\sigma_x, \sigma_y, \sigma_{\cos}, \sigma_{\sin}) \in \mathbb{R}^{4 \times 4}.
$$

当前配置中：

$$
\mu = (0,0,0,0), \qquad
(\sigma_x, \sigma_y, \sigma_{\cos}, \sigma_{\sin}) = (0.5, 0.5, 1, 1).
$$

真实动作的归一化形式为

$$
Z_0 = (A_0 - \mu)\,S^{-1} \in \mathbb{R}^{T \times 4}.
$$

模型直接预测归一化动作 $Z_\theta \in \mathbb{R}^{T \times 4}$。定义归一化空间中的误差：

$$
E = Z_\theta - Z_0 \in \mathbb{R}^{T \times 4}.
$$

将 $E$ 按列拆分：

$$
E =
\begin{bmatrix}
e_x & e_y & e_{\cos} & e_{\sin}
\end{bmatrix},
$$

其中 $e_x, e_y, e_{\cos}, e_{\sin} \in \mathbb{R}^T$ 分别是四个维度的误差向量。

---

## 3. Velocity Loss（直接监督位移增量）

在归一化空间中对全部四个维度计算均方误差（MSE）：

$$
\mathcal{L}_{\text{velocity}} = \frac{1}{T} \|E\|_F^2.
$$

Frobenius 范数平方等于所有元素的平方和：

$$
\|E\|_F^2 = \sum_{t=1}^{T} \sum_{d=1}^{4} E_{t,d}^2.
$$

按维度展开：

$$
\mathcal{L}_{\text{velocity}} = \frac{1}{T} \left( \sum_{t=1}^T e_{x,t}^2 + \sum_{t=1}^T e_{y,t}^2 + \sum_{t=1}^T e_{\cos,t}^2 + \sum_{t=1}^T e_{\sin,t}^2 \right),
$$

写成向量内积形式：

$$
\mathcal{L}_{\text{velocity}} = \frac{1}{T} \left( e_x^\top e_x + e_y^\top e_y + e_{\cos}^\top e_{\cos} + e_{\sin}^\top e_{\sin} \right).
$$

---

## 4. 积分矩阵 $M$

位移增量需要累加才能得到绝对位置。定义下三角积分矩阵 $M \in \mathbb{R}^{T \times T}$：

$$
M_{ij} = \begin{cases}
1, & i \ge j, \\
0, & i < j.
\end{cases}
$$

即

$$
M =
\begin{bmatrix}
1 & 0 & 0 & \cdots & 0 \\
1 & 1 & 0 & \cdots & 0 \\
1 & 1 & 1 & \cdots & 0 \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & 1 & 1 & \cdots & 1
\end{bmatrix}.
$$

例如 $T=3$ 时：

$$
M =
\begin{bmatrix}
1 & 0 & 0 \\
1 & 1 & 0 \\
1 & 1 & 1
\end{bmatrix}.
$$

对于位移增量向量 $a = (\Delta x_1, \Delta x_2, \dots, \Delta x_T)^\top$，绝对位置为

$$
x = M a =
\begin{bmatrix}
\Delta x_1 \\
\Delta x_1 + \Delta x_2 \\
\vdots \\
\Delta x_1 + \Delta x_2 + \cdots + \Delta x_T
\end{bmatrix}.
$$

---

## 5. 反归一化后的物理误差

模型预测的动作反归一化为物理尺度：

$$
A_\theta = Z_\theta\,S + \mu.
$$

真实动作同样满足 $A_0 = Z_0\,S + \mu$。因此物理动作误差为

$$
A_\theta - A_0 = (Z_\theta - Z_0)\,S = E\,S.
$$

由于 $S$ 是对角矩阵，按维度写出：

- x 方向的位移增量误差：$\delta a_x = \sigma_x e_x$（一个 $T$ 维向量）
- y 方向的位移增量误差：$\delta a_y = \sigma_y e_y$
- 航向角正余弦的误差：$\delta a_{\cos} = \sigma_{\cos} e_{\cos} = e_{\cos}$（因为 $\sigma_{\cos}=1$），$\delta a_{\sin} = e_{\sin}$。

---

## 6. Waypoint Loss（间接监督绝对位置）

将物理位移增量误差积分得到位置误差：

$$
\delta x = M(\sigma_x e_x), \qquad \delta y = M(\sigma_y e_y).
$$

其中 $\delta x, \delta y \in \mathbb{R}^T$ 是每个时刻的 x/y 位置误差。

Waypoint loss 定义为这两个误差的均方和：

$$
\mathcal{L}_{\text{waypoint}} = \frac{1}{T} \left( \|\delta x\|_2^2 + \|\delta y\|_2^2 \right).
$$

展开第一个范数：

$$
\|\delta x\|_2^2 = (M \sigma_x e_x)^\top (M \sigma_x e_x)
= \sigma_x^2 e_x^\top M^\top M e_x.
$$

同理

$$
\|\delta y\|_2^2 = \sigma_y^2 e_y^\top M^\top M e_y.
$$

因此

$$
\mathcal{L}_{\text{waypoint}}
= \frac{1}{T} \left( \sigma_x^2 e_x^\top M^\top M e_x + \sigma_y^2 e_y^\top M^\top M e_y \right).
$$

**逐时间步展开（以 x 方向为例）**：  
$M e_x$ 的第 $t$ 个元素为 $\sum_{i=1}^t e_{x,i}$，所以

$$
\|M e_x\|_2^2 = \sum_{t=1}^T \left( \sum_{i=1}^t e_{x,i} \right)^2.
$$

这清楚地表明 waypoint loss 是对累积误差的惩罚。

---

## 7. 合并为 P 范数

Hybrid loss 定义为

$$
\mathcal{L}_{\text{hybrid}} = \mathcal{L}_{\text{velocity}} + \omega \mathcal{L}_{\text{waypoint}}.
$$

代入前面的表达式：

$$
\begin{aligned}
\mathcal{L}_{\text{hybrid}}
=& \frac{1}{T} \big[
e_x^\top e_x + e_y^\top e_y + e_{\cos}^\top e_{\cos} + e_{\sin}^\top e_{\sin} \\
&\quad + \omega \sigma_x^2 e_x^\top M^\top M e_x + \omega \sigma_y^2 e_y^\top M^\top M e_y \big].
\end{aligned}
$$

对 $x$ 和 $y$ 维度分别合并同类项：

$$
e_x^\top e_x + \omega \sigma_x^2 e_x^\top M^\top M e_x
= e_x^\top \left( I + \omega \sigma_x^2 M^\top M \right) e_x.
$$

定义

$$
P_x = I + \omega \sigma_x^2 M^\top M, \qquad
P_y = I + \omega \sigma_y^2 M^\top M.
$$

则

$$
\mathcal{L}_{\text{hybrid}}
= \frac{1}{T} \left(
e_x^\top P_x e_x + e_y^\top P_y e_y + e_{\cos}^\top I e_{\cos} + e_{\sin}^\top I e_{\sin}
\right).
$$

若将误差按坐标维度优先排列成一个大向量：

$$
e = \begin{bmatrix} e_x \\ e_y \\ e_{\cos} \\ e_{\sin} \end{bmatrix} \in \mathbb{R}^{4T},
$$

并定义块对角矩阵

$$
P_{\text{full}} =
\begin{bmatrix}
P_x & 0 & 0 & 0 \\
0 & P_y & 0 & 0 \\
0 & 0 & I & 0 \\
0 & 0 & 0 & I
\end{bmatrix}
\in \mathbb{R}^{4T \times 4T},
$$

则混合损失可以简洁地写成

$$
\mathcal{L}_{\text{hybrid}} = \frac{1}{T} e^\top P_{\text{full}}\, e = \frac{1}{T} \|e\|_{P_{\text{full}}}^2.
$$

---

## 8. $P$ 的正定性

对任意非零向量 $z \in \mathbb{R}^T$：

$$
z^\top P_x z = z^\top z + \omega \sigma_x^2 z^\top M^\top M z
= \|z\|_2^2 + \omega \sigma_x^2 \|M z\|_2^2.
$$

由于 $\omega \ge 0$ 且 $\sigma_x > 0$，第一项 $\|z\|_2^2 > 0$，故 $z^\top P_x z > 0$。因此 $P_x \succ 0$，同理 $P_y \succ 0$，$P_{\text{full}}$ 正定。

**理论意义**：正定二次型是 Bregman 散度，其条件回归的最优解为条件期望 $\mathbb{E}[Y|X]$。在扩散模型中，这意味着 hybrid loss 不会改变理论最优预测，仍然能恢复正确的 marginal score function。

---

## 9. 完整积分的梯度累积问题

考虑 waypoint loss 对归一化误差 $e_x$ 的梯度。由

$$
\mathcal{L}_{\text{wp},x} = \frac{\sigma_x^2}{T} e_x^\top M^\top M e_x
$$

可得

$$
\frac{\partial \mathcal{L}_{\text{wp},x}}{\partial e_x}
= \frac{2\sigma_x^2}{T} M^\top M e_x.
$$

记 $r = M(\sigma_x e_x)$ 为位置残差向量，则第 $j$ 个增量的梯度为：

$$
\left( \frac{\partial \mathcal{L}_{\text{wp},x}}{\partial e_x} \right)_j
= \frac{2\sigma_x}{T} \sum_{t=j}^{T} r_t.
$$

**解释**：第 $j$ 个位移增量会影响所有 $t \ge j$ 的绝对位置，因此它接收从第 $j$ 帧到最后一帧的所有位置残差之和。越早的增量，梯度累积越大；越晚的增量，梯度越小。这导致训练时梯度分布严重不平衡。

---

## 10. 截断梯度（Detached Integral）

为了缓解梯度不平衡，引入一个窗口大小 $W$。在前向计算时保持完整积分，但在反向传播时只允许梯度回传到最近 $W$ 个位移增量上。具体地，对于第 $t$ 个位置，我们写成：

$$
\widetilde{x}_t = \text{sg}\left( \sum_{i=1}^{t-W} a_i \right) + \sum_{i=\max(1,t-W+1)}^{t} a_i,
$$

其中 $\text{sg}(\cdot)$ 表示 stop-gradient（前向值不变，反向梯度为 0）。

前向数值与完整积分完全一致：

$$
\widetilde{x}_t = \sum_{i=1}^{t} a_i = x_t.
$$

但反向传播时，雅可比矩阵变为：

$$
\frac{\partial \widetilde{x}_t}{\partial a_j} =
\begin{cases}
1, & \max(1, t-W+1) \le j \le t, \\
0, & \text{其他}.
\end{cases}
$$

**举例**：设 $T=5, W=2$，则

$$
\begin{aligned}
\widetilde{x}_1 &= a_1, \\
\widetilde{x}_2 &= a_1 + a_2, \\
\widetilde{x}_3 &= \text{sg}(a_1) + a_2 + a_3, \\
\widetilde{x}_4 &= \text{sg}(a_1 + a_2) + a_3 + a_4, \\
\widetilde{x}_5 &= \text{sg}(a_1 + a_2 + a_3) + a_4 + a_5.
\end{aligned}
$$

最后一个位置虽然前向使用了全部五个增量，但梯度只会流向 $a_4$ 和 $a_5$。

---

## 11. 代码中前向精确、反向代理的实现

在 `hybrid_loss_quadratic.py` 中，使用以下技巧：

```python
loss = (
    quadratic_loss.detach()      # 前向精确值，但不提供梯度
    + surrogate_loss              # 提供代理梯度
    - surrogate_loss.detach()     # 前向抵消，不影响数值
)
```

设 $L_q$ 为完整二次型损失，$L_s$ 为带截断积分的代理损失。则上述表达式的前向数值为：

$$
\text{sg}(L_q) + L_s - \text{sg}(L_s) = L_q.
$$

反向梯度为：

$$
\nabla \text{loss} = 0 + \nabla L_s - 0 = \nabla L_s.
$$

因此，日志和评估使用的损失数值是精确的二次型，而优化器实际接收的梯度来自窗口截断的代理损失。

---

## 12. 最终公式与 reduction

设 batch size 为 $B$，对每个样本 $b$，其损失为

$$
\mathcal{L}_b = \frac{1}{T} \left(
e_{b,x}^\top P_x e_{b,x} + e_{b,y}^\top P_y e_{b,y} + \|e_{b,\cos}\|_2^2 + \|e_{b,\sin}\|_2^2
\right).
$$

总损失为 batch 上的平均：

$$
\mathcal{L} = \frac{1}{B} \sum_{b=1}^B \mathcal{L}_b
= \frac{1}{BT} \sum_{b=1}^B \left( e_{b,x}^\top P_x e_{b,x} + e_{b,y}^\top P_y e_{b,y} + \|e_{b,\cos}\|_2^2 + \|e_{b,\sin}\|_2^2 \right).
$$

当前配置下 $\sigma_x = \sigma_y = 0.5$，$\omega = 0.01$，因此

$$
P_x = P_y = I + 0.0025 M^\top M.
$$

训练时默认使用窗口 $W=10$ 的截断梯度（对应 10 Hz 采样率下约 1 秒的时间窗口）。

---

## 13. 总结

- **混合损失**：速度（位移增量）MSE + 加权位置 MSE。
- **二次型**：合并后为 $P$ 范数，其中 $P$ 正定。
- **理论保证**：正定 $P$ 不改变扩散模型的最优解。
- **工程稳定**：窗口截断梯度平衡时间维度的梯度传播。

如果模型输出扩展为 `(dx, dy, cosθ, sinθ, vx, vy)`（即增加物理速度 vx, vy），混合损失可以按以下方式自然扩展。

---

## 1. 输出与误差定义

模型输出维度变为 $D=6$，在归一化空间中预测记为 $Z_\theta \in \mathbb{R}^{B \times T \times 6}$。

对应的真实目标 $A_0$ 包含：
- 位移增量 $\Delta x_t, \Delta y_t$
- 航向 $\cos\theta_t, \sin\theta_t$
- 速度 $v_{x,t}, v_{y,t}$（物理速度，单位 m/s）

同样对每个维度做归一化（使用各自的均值、标准差），得到 $Z_0$。  
归一化误差：

$$
E = Z_\theta - Z_0 \in \mathbb{R}^{B \times T \times 6}
$$

按维度拆分：

$$
E = \begin{bmatrix}
e_x & e_y & e_{\cos} & e_{\sin} & e_{vx} & e_{vy}
\end{bmatrix}
$$

其中 $e_x, e_y \in \mathbb{R}^T$ 对应位移增量误差，$e_{vx}, e_{vy}$ 对应速度误差。

---

## 2. 损失函数扩展

### 2.1 Velocity Loss（直接监督所有输出维度）

对全部 6 个维度计算 MSE：

$$
\mathcal{L}_{\text{velocity}} = \frac{1}{T} \left(
e_x^\top e_x + e_y^\top e_y + e_{\cos}^\top e_{\cos} + e_{\sin}^\top e_{\sin} + e_{vx}^\top e_{vx} + e_{vy}^\top e_{vy}
\right)
$$

### 2.2 Waypoint Loss（仍只由位移增量积分得到位置）

位移增量部分仍为前两个维度 $e_x, e_y$，与之前完全相同：

$$
\mathcal{L}_{\text{waypoint}} = \frac{1}{T} \left(
\sigma_x^2 e_x^\top M^\top M e_x + \sigma_y^2 e_y^\top M^\top M e_y
\right)
$$

速度 $v_x, v_y$ 不参与积分，因此 waypoint loss 不受影响。

### 2.3 Hybrid Loss

$$
\mathcal{L}_{\text{hybrid}} = \mathcal{L}_{\text{velocity}} + \omega \mathcal{L}_{\text{waypoint}}
$$

合并后写成 P 范数形式：

$$
\mathcal{L}_{\text{hybrid}} = \frac{1}{T} \left(
e_x^\top P_x e_x + e_y^\top P_y e_y + e_{\cos}^\top I e_{\cos} + e_{\sin}^\top I e_{\sin} + e_{vx}^\top I e_{vx} + e_{vy}^\top I e_{vy}
\right)
$$

其中

$$
P_x = I + \omega \sigma_x^2 M^\top M,\qquad
P_y = I + \omega \sigma_y^2 M^\top M
$$

与之前完全一致。

若将误差堆叠为长向量 $e = [e_x; e_y; e_{\cos}; e_{\sin}; e_{vx}; e_{vy}] \in \mathbb{R}^{6T}$，则块对角矩阵为

$$
P_{\text{full}} = \operatorname{blockdiag}(P_x, P_y, I, I, I, I)
$$

最终

$$
\mathcal{L}_{\text{hybrid}} = \frac{1}{T} e^\top P_{\text{full}}\, e
$$

---

## 3. 代码修改要点

在原 `quadratic_hybrid_loss` 的基础上，主要修改：

- **维度处理**：  
  - 输入 `prediction` 和 `ego_future` 的形状从 `(B,T,4)` 变为 `(B,T,6)`。  
  - `waypoint_to_action` 需要额外计算速度，可以从绝对位置差分后除以 `dt`，或直接从数据集中读取（如果提供）。

- **误差拆分**：  
  ```python
  error_xy   = error[:, :, :2]      # dx, dy
  error_head = error[:, :, 2:4]     # cos, sin
  error_vel  = error[:, :, 4:]      # vx, vy
  ```

- **归一化参数**：  
  `mean` 和 `std` 长度变为 6，需提供 `vx, vy` 的均值和标准差。

- **P 矩阵构造**：  
  不变，仍然只针对 x/y 维度构造 `metric_p`。

- **二次型计算**：  
  - 位置二次型：`error_xy` 与 `metric_p` 相乘（同原代码）。  
  - 航向与速度二次型：直接对 `error_head` 和 `error_vel` 做平方和。

- **日志项**：  
  `velocity_loss` 现在包含所有 6 维的 MSE；`waypoint_loss` 不变。

- **截断梯度**：  
  仍然只对 `predicted_action` 的前两维（位移增量）做 `detached_integral`。

---

## 4. 注意事项

- **物理一致性**：如果 `dx, dy` 和 `vx, vy` 之间存在线性关系（$dx = vx \cdot dt$），模型可能学到冗余信息，但不影响损失计算。若需要显式约束，可添加一致性项，如 $\|dx - vx\cdot dt\|^2$。
- **归一化**：新增的速度维度需合理设置标准差（例如物理速度的标准差），避免某一维主导梯度。
- **扩展性**：上述方法可推广到任意数量的辅助输出维度，只需将对应误差加入 velocity loss 并使用单位矩阵（或自定义正定矩阵）作为权重。

这样，混合损失框架被平滑地扩展到了 6 维输出，同时保持了原有理论性质（正定 P 矩阵、Bregman 散度等）。

In [ ]:
from typing import Dict, Optional

import torch
from torch import Tensor
import json
from mmengine import fileio

def openjson(path):
    """读取 JSON 文件并返回解析后的字典"""
    value = fileio.get_text(path)
    dict_data = json.loads(value)
    return dict_data

class StateNormalizer:
    """状态归一化器：提供归一化和反归一化操作，用于将物理量转换到归一化空间（或反之）"""
    def __init__(self, mean, std):
        # mean, std 形状都是 (4,)，对应 (dx, dy, cos, sin) 四个维度
        self.mean = torch.as_tensor(mean)
        self.std = torch.as_tensor(std)

    @classmethod
    def from_json(cls, args):
        """从配置文件（normalization.json）中读取均值和标准差，构建实例"""
        data = openjson(args.normalization_file_path)
        mean = [data["ego"]["mean"]]      # 例如 [0,0,0,0]
        std = [data["ego"]["std"]]        # 例如 [0.5,0.5,1,1]
        return cls(mean, std)
    
    def __call__(self, data):
        """归一化：z = (x - mean) / std
        data 形状 (..., 4)，返回同形状张量"""
        return (data - self.mean.to(data.device)) / self.std.to(data.device)

    def inverse(self, data):
        """反归一化：x = z * std + mean
        data 形状 (..., 4)，返回同形状张量"""
        return data * self.std.to(data.device) + self.mean.to(data.device)

    def to_dict(self):
        """导出为字典（用于保存配置）"""
        return {
            "mean": self.mean.detach().cpu().numpy().tolist(),
            "std": self.std.detach().cpu().numpy().tolist()
        }

In [ ]:
from typing import Dict, Optional

import torch
from torch import Tensor


def waypoint_to_action_general(ego_future: Tensor, n_integral: int = 2) -> Tensor:
    """将绝对轨迹 (B, T, D) 转换为动作表示 (B, T, D)。

    前 n_integral 列被视为位置（通常为 x, y），做差分得到位移增量；
    其余列（如航向、速度等）保持不变。

    参数:
        ego_future: (B, T, D) 未归一化的绝对轨迹，前 n_integral 列为绝对位置。
        n_integral: 需要做差分的前几列数量，通常为 2（对应 x, y）。

    返回:
        action: (B, T, D) 动作表示，前 n_integral 列为位移增量，其余列不变。
    """
    D = ego_future.shape[-1]
    if n_integral > D:
        raise ValueError(f"n_integral ({n_integral}) cannot exceed total dimensions ({D})")

    # 分离积分部分（位置）和非积分部分（航向、速度等）
    integral_part = ego_future[:, :, :n_integral]          # (B, T, n_integral)
    non_integral_part = ego_future[:, :, n_integral:]      # (B, T, D - n_integral)

    # 构造前一帧的位置：在时间维开头补一个零点，然后取 [:, :-1]
    origin = torch.zeros_like(integral_part[:, :1])        # (B, 1, n_integral) 初始位置 (0,0,...)
    previous = torch.cat([origin, integral_part[:, :-1]], dim=1)  # (B, T, n_integral)
    delta = integral_part - previous                       # (B, T, n_integral) 位移增量

    # 将差分后的位移增量与非积分部分拼接，恢复原始维度
    return torch.cat([delta, non_integral_part], dim=-1)   # (B, T, D)


def detached_integral(delta: Tensor, window_size: int) -> Tensor:
    """对位移增量进行时间积分（累积求和），但反向传播时只让梯度通过最近 window_size 个时间步。

    前向输出与普通 cumsum 完全一致，只是梯度被截断。

    参数:
        delta: (B, T, D) 每帧的位移增量（可以是 x,y 或其他维度）
        window_size: 梯度回传的窗口大小（正整数）

    返回:
        integrated: (B, T, D) 积分后的绝对位置（前向与 cumsum 相同）
    """
    if window_size < 1:
        raise ValueError(f"window_size must be positive, got {window_size}")

    horizon = delta.shape[1]                # 时间长度 T
    window = min(window_size, horizon)      # 实际窗口长度不能超过 T

    # 完整累积和 C[t] = sum_{i=1}^t delta[i]
    full_sum = delta.cumsum(dim=1)          # (B, T, D)

    if window == horizon:
        # 窗口覆盖整个时域，无需截断，直接返回（梯度全部保留）
        return full_sum

    # ---------- 构造最近 window 步的和（保留梯度） ----------
    # recent 的每一行 = C[t] - C[t-window]，只包含最近 window 个增量的和
    # 对于 t < window，直接取 C[t]（因为此时 t-window < 0）
    recent = torch.cat(
        [
            full_sum[:, :window],                                # 前 window 步直接取累积和
            full_sum[:, window:] - full_sum[:, :-window],        # 对于 t >= window，C[t] - C[t-window]
        ],
        dim=1,
    )                                                           # (B, T, D)

    # ---------- 构造窗口之前的历史（梯度截断） ----------
    # detached_history 对应 C[t-window]，但使用 detach 切断梯度
    detached_history = torch.cat(
        [
            torch.zeros_like(full_sum[:, :window]),             # 前 window 步没有历史，用 0 填充
            full_sum[:, :-window].detach(),                      # 对于 t >= window，历史部分 C[t-window]，梯度被截断
        ],
        dim=1,
    )                                                           # (B, T, D)

    # 前向：detached_history + recent = full_sum（数值完全一致）
    # 反向：梯度只通过 recent 部分，因此只回传到最近 window 个 delta
    return detached_history + recent                            # (B, T, D)


def quadratic_hybrid_loss_general(
    prediction: Tensor,
    ego_future: Tensor,
    norm,
    *,
    omega: float = 0.01,
    detach_window_size: Optional[int] = 10,
    n_integral: int = 2,
) -> Dict[str, Tensor]:
    """通用二次型混合损失，支持任意维度输出。

    参数:
        prediction: (B, T, D) 模型预测，归一化后的动作表示（例如 dx,dy,cos,sin,...）
        ego_future: (B, T, D) 真实轨迹，未归一化的绝对量（例如 x,y,cos,sin,...）
        norm: StateNormalizer 实例，其 mean/std 长度必须等于 D
        omega: waypoint loss 权重（默认 0.01）
        detach_window_size: 截断梯度窗口大小，None 表示不截断
        n_integral: 前几个维度作为位移增量参与积分（通常为 2，即 dx,dy）

    返回:
        字典，包含以下键：
            loss: 用于反向传播的训练损失（可能是精确二次型或带截断的代理损失）
            velocity_loss: 归一化空间中的全部维度 MSE
            waypoint_loss: 物理位移增量积分后的位置 MSE
            quadratic_loss: 精确的二次型损失数值
            metric_p: 形状为 (n_integral, T, T) 的 P 矩阵
    """
    D = prediction.shape[-1]
    if prediction.ndim != 3 or D != ego_future.shape[-1]:
        raise ValueError("prediction and ego_future must have shape (B, T, D) with same D")
    if n_integral > D:
        raise ValueError(f"n_integral ({n_integral}) cannot exceed total dimensions ({D})")
    if omega < 0:
        raise ValueError("omega must be non-negative")
    if detach_window_size is not None and detach_window_size < 1:
        raise ValueError("detach_window_size must be positive or None")

    batch_size, horizon, _ = prediction.shape   # B, T

    # ---------- 准备归一化参数 ----------
    # mean, std 形状 (D,)，转换为与 prediction 相同的设备和数据类型
    mean = norm.mean.to(prediction).reshape(D)   # (D,)
    std = norm.std.to(prediction).reshape(D)     # (D,)

    # ---------- 1. 构造归一化空间中的误差 E = Z_θ - Z_0 ----------
    # 将真实绝对轨迹转换为动作表示（物理尺度）
    target_action = waypoint_to_action_general(ego_future, n_integral)  # (B, T, D)
    # 对动作进行归一化
    target_normalized = (target_action - mean) / std                    # (B, T, D)
    # 计算归一化误差
    error = prediction - target_normalized                              # (B, T, D)

    # 拆分误差：积分部分（前 n_integral 列）和非积分部分
    error_integral = error[:, :, :n_integral]             # (B, T, n_integral)
    error_non_integral = error[:, :, n_integral:]         # (B, T, D - n_integral)

    # 积分部分对应的标准差
    std_integral = std[:n_integral]                       # (n_integral,)

    # ---------- 2. 构造 P 矩阵（仅针对积分部分） ----------
    # M 为下三角积分矩阵 (T, T)，M_{ij}=1 if i>=j else 0
    matrix_m = torch.tril(prediction.new_ones(horizon, horizon))  # (T, T)
    # Gram 矩阵 M^T M
    gram = matrix_m.T @ matrix_m                                   # (T, T)
    # 单位矩阵 I
    identity = torch.eye(horizon, device=prediction.device, dtype=prediction.dtype)  # (T, T)

    # metric_p 形状 (n_integral, T, T)
    # metric_p[d] = I + omega * std_integral[d]^2 * M^T M
    # 通过广播 std_integral.square()[:, None, None] 将 (n_integral,) 扩展为 (n_integral,1,1)
    metric_p = identity[None] + (
        omega * std_integral.square()[:, None, None] * gram[None]
    )                                                           # (n_integral, T, T)

    # ---------- 3. 计算二次型 e_d^T P_d e_d（积分部分） ----------
    # 将误差转置为 (B, n_integral, T) 以便与 metric_p 进行矩阵乘法
    error_integral_t = error_integral.transpose(1, 2)     # (B, n_integral, T)
    # 增加一个维度用于 @ 运算：变为 (B, n_integral, T, 1)
    error_integral_col = error_integral_t.unsqueeze(-1)   # (B, n_integral, T, 1)

    # 计算 P_d @ e_d：metric_p[None] 形状 (1, n_integral, T, T)，与 (B, n_integral, T, 1) 广播相乘
    weighted_error_integral = (
        metric_p[None] @ error_integral_col
    ).squeeze(-1)                                         # (B, n_integral, T) -> 得到 P_d e_d

    # 二次型：e_d^T P_d e_d = e_d * (P_d e_d)，然后对坐标维度和时间维度求和
    integral_quadratic = (
        error_integral_t * weighted_error_integral
    ).sum(dim=(1, 2))                                     # (B,) 每个样本的积分部分二次型值

    # 非积分部分使用单位矩阵（普通 MSE），直接平方求和
    non_integral_quadratic = error_non_integral.square().sum(dim=(1, 2))  # (B,)

    # 总二次型损失：先对 batch 求和，再除以 (B*T)
    quadratic_loss = (integral_quadratic + non_integral_quadratic).sum() / (batch_size * horizon)  # 标量

    # ---------- 4. 单独计算原始两项，用于日志和验证等价性 ----------
    # velocity loss: 所有维度的 MSE
    velocity_loss = error.square().sum() / (batch_size * horizon)  # 标量

    # waypoint loss: 仅由积分部分积分得到
    # 归一化误差乘以标准差恢复物理尺度
    physical_delta_error = error_integral * std_integral  # (B, T, n_integral) 物理位移增量误差
    # 用矩阵 M 积分得到位置误差
    # matrix_m 形状 (T, T)，physical_delta_error 形状 (B, T, n_integral)，广播相乘
    waypoint_error = matrix_m @ physical_delta_error      # (B, T, n_integral) 位置误差
    waypoint_loss = waypoint_error.square().sum() / (batch_size * horizon)  # 标量

    # 理论上 quadratic_loss == velocity_loss + omega * waypoint_loss

    # ---------- 5. 可选：前向用二次型数值，反向用窗口截断梯度 ----------
    if detach_window_size is None:
        # 不使用截断，直接使用二次型及其标准梯度
        loss = quadratic_loss
    else:
        # 反归一化模型预测，得到物理动作
        predicted_action = prediction * std + mean        # (B, T, D)
        # 对物理动作的积分维度做带截断的积分
        predicted_waypoint = detached_integral(
            predicted_action[:, :, :n_integral], detach_window_size
        )                                                  # (B, T, n_integral)

        # 真实位置：直接从输入的绝对轨迹中取前 n_integral 列
        target_waypoint = ego_future[:, :, :n_integral]    # (B, T, n_integral)
        # 基于截断积分的 waypoint loss
        windowed_waypoint_loss = (
            predicted_waypoint - target_waypoint
        ).square().sum() / (batch_size * horizon)          # 标量

        # 代理损失 = velocity loss + omega * windowed waypoint loss
        surrogate_loss = velocity_loss + omega * windowed_waypoint_loss

        # 前向：quadratic_loss；反向：surrogate_loss 的梯度
        # 通过 detach 技巧实现前向和反向分离
        loss = (
            quadratic_loss.detach()            # 前向值参与，但不产生梯度
            + surrogate_loss                   # 提供代理梯度
            - surrogate_loss.detach()          # 前向抵消
        )

    # ---------- 返回结果 ----------
    return {
        "loss": loss,                              # 用于训练（可能带截断梯度）
        "velocity_loss": velocity_loss.detach(),   # 仅日志
        "waypoint_loss": waypoint_loss.detach(),   # 仅日志
        "quadratic_loss": quadratic_loss.detach(), # 精确二次型数值
        "metric_p": metric_p.detach(),             # (n_integral, T, T) P 矩阵
    }


# ---------- 便捷包装函数 ----------
def quadratic_hybrid_loss_4d(
    prediction: Tensor,
    ego_future: Tensor,
    norm,
    *,
    omega: float = 0.01,
    detach_window_size: Optional[int] = 10,
) -> Dict[str, Tensor]:
    """4 维输出：(dx, dy, cosθ, sinθ) 的混合损失。"""
    return quadratic_hybrid_loss_general(
        prediction, ego_future, norm,
        omega=omega,
        detach_window_size=detach_window_size,
        n_integral=2,
    )


def quadratic_hybrid_loss_6d(
    prediction: Tensor,
    ego_future: Tensor,
    norm,
    *,
    omega: float = 0.01,
    detach_window_size: Optional[int] = 10,
) -> Dict[str, Tensor]:
    """6 维输出：(dx, dy, cosθ, sinθ, vx, vy) 的混合损失。"""
    return quadratic_hybrid_loss_general(
        prediction, ego_future, norm,
        omega=omega,
        detach_window_size=detach_window_size,
        n_integral=2,
    )

In [ ]:
# 假设已有一个支持任意维度的 StateNormalizer 类
class StateNormalizer:
    def __init__(self, mean, std):
        self.mean = torch.tensor(mean)
        self.std = torch.tensor(std)
    def __call__(self, data):
        return (data - self.mean.to(data.device)) / self.std.to(data.device)
    def inverse(self, data):
        return data * self.std.to(data.device) + self.mean.to(data.device)

# 4 维示例
norm_4d = StateNormalizer(mean=[0,0,0,0], std=[0.5,0.5,1,1])
pred_4d = torch.randn(2, 10, 4)  # 归一化后的预测
gt_4d = torch.randn(2, 10, 4)    # 绝对轨迹 (x,y,cos,sin)
result_4d = quadratic_hybrid_loss_4d(pred_4d, gt_4d, norm_4d, detach_window_size=10)
loss_4d = result_4d["loss"]
loss_4d.backward()

# 6 维示例
norm_6d = StateNormalizer(mean=[0,0,0,0,0,0], std=[0.5,0.5,1,1,2.0,2.0])
pred_6d = torch.randn(2, 10, 6)  # 归一化后的预测 (dx,dy,cos,sin,vx,vy)
gt_6d = torch.randn(2, 10, 6)    # 绝对轨迹 (x,y,cos,sin,vx,vy)
result_6d = quadratic_hybrid_loss_6d(pred_6d, gt_6d, norm_6d, detach_window_size=10)
loss_6d = result_6d["loss"]
loss_6d.backward()

# Hybrid Loss 代码核心解析

## 1. 归一化器 `StateNormalizer`

```python
def __call__(self, data):
    return (data - self.mean.to(data.device)) / self.std.to(data.device)

def inverse(self, data):
    return data * self.std.to(data.device) + self.mean.to(data.device)
```

**说明**：  
- 归一化：$Z = (A - \mu) / \sigma$；反归一化：$A = Z \cdot \sigma + \mu$。  
- `mean` 和 `std` 为形状 `(4,)` 的张量，当前配置 $\mu=(0,0,0,0)$，$\sigma=(0.5,0.5,1,1)$。  
- 在损失计算中用于将真实动作转换为归一化目标，以及将模型预测恢复为物理尺度。

---

## 2. 绝对轨迹 → 动作表示 `waypoint_to_action`

```python
xy = ego_future[:, :, :2]                # (B,T,2)
origin = torch.zeros_like(xy[:, :1])     # (B,1,2)
previous_xy = torch.cat([origin, xy[:, :-1]], dim=1)
delta_xy = xy - previous_xy              # 位移增量
return torch.cat([delta_xy, heading], dim=-1)
```

**说明**：  
- 前两维（位置）做差分：$\Delta x_t = x_t - x_{t-1}$，$\Delta y_t = y_t - y_{t-1}$，并令 $x_0=y_0=0$。  
- 后两维（航向正余弦）直接保留，不做差分。  
- 输出形状 `(B,T,4)`，对应动作目标 $A_0$。

---

## 3. 带截断梯度的积分 `detached_integral`

```python
full_sum = delta.cumsum(dim=1)                           # 完整累积和
recent = torch.cat([
    full_sum[:, :window],
    full_sum[:, window:] - full_sum[:, :-window],
], dim=1)
detached_history = torch.cat([
    torch.zeros_like(full_sum[:, :window]),
    full_sum[:, :-window].detach(),
], dim=1)
return detached_history + recent                         # 前向等于 full_sum
```

**说明**：  
- 前向值：`detached_history + recent = full_sum`，与普通 `cumsum` 完全相同。  
- 反向梯度：`recent` 部分保留了最近 `window` 个时间步的梯度；`detached_history` 通过 `.detach()` 切断了更早历史的梯度。  
- 效果：每个位置的梯度只回传到最近 $W$ 个位移增量，缓解积分导致的梯度不平衡。  
- 数学对应：
  $$
  \widetilde{x}_t = \operatorname{sg}\left(\sum_{i=1}^{t-W} a_i\right) + \sum_{i=t-W+1}^{t} a_i
  $$

---

## 4. 构造归一化误差

```python
target_action = waypoint_to_action(ego_future)          # 物理动作
target_normalized = (target_action - mean) / std        # 归一化目标
error = prediction - target_normalized                  # 归一化误差 E
error_xy = error[:, :, :2]                              # e_x, e_y
error_heading = error[:, :, 2:]                         # e_cos, e_sin
```

**说明**：  
- 误差在归一化空间中计算，形状 `(B,T,4)`。  
- 拆分为位置误差 `error_xy`（两个通道）和航向误差 `error_heading`。

---

## 5. 构造 $P$ 矩阵

```python
matrix_m = torch.tril(prediction.new_ones(horizon, horizon))   # M
gram = matrix_m.T @ matrix_m                                    # M^T M
metric_p = identity[None] + (
    omega * std_xy.square()[:, None, None] * gram[None]
)                                                               # (2,T,T)
```

**说明**：  
- `matrix_m` 是下三角全 1 矩阵（积分矩阵 $M$）。  
- `gram = M^\top M`。  
- `metric_p` 为 $P_x, P_y$ 的堆叠，其中  
  $P_d = I + \omega \sigma_d^2 M^\top M, \quad d \in \{x,y\}$。  
- 通过广播 `std_xy.square()[:, None, None]` 分别对 x/y 维度缩放，无需显式构造 $4T\times4T$ 的大矩阵。

---

## 6. 二次型计算

```python
error_xy = error_xy.transpose(1, 2)                    # (B,2,T)
weighted_error_xy = (metric_p[None] @ error_xy.unsqueeze(-1)).squeeze(-1)
position_quadratic = (error_xy * weighted_error_xy).sum(dim=(1,2))
heading_quadratic = error_heading.square().sum(dim=(1,2))
quadratic_loss = (position_quadratic + heading_quadratic).sum() / (batch_size*horizon)
```

**说明**：  
- `weighted_error_xy` 计算 $P_d e_d$。  
- `position_quadratic` 计算 $e_x^\top P_x e_x + e_y^\top P_y e_y$。  
- `heading_quadratic` 计算 $\|e_{\cos}\|^2 + \|e_{\sin}\|^2$。  
- 最终除以 `batch_size * horizon`，与原始 `mean` 缩减一致。

---

## 7. 原始两项损失（用于日志与验证）

```python
velocity_loss = error.square().sum() / (batch_size * horizon)

physical_delta_error = error[:, :, :2] * std_xy           # 恢复物理尺度
waypoint_error = matrix_m @ physical_delta_error           # 积分
waypoint_loss = waypoint_error.square().sum() / (batch_size * horizon)
```

**说明**：  
- `velocity_loss`：归一化空间中四维 MSE。  
- `waypoint_loss`：物理位移误差积分后的位置 MSE。  
- 两者满足：$\texttt{quadratic\_loss} = \texttt{velocity\_loss} + \omega \cdot \texttt{waypoint\_loss}$。

---

## 8. 截断梯度的组合技巧

```python
if detach_window_size is None:
    loss = quadratic_loss
else:
    predicted_action = prediction * std + mean            # 反归一化
    predicted_waypoint = detached_integral(predicted_action[:, :, :2], detach_window_size)
    windowed_waypoint_loss = (predicted_waypoint - ego_future[:, :, :2]).square().sum() / (batch_size*horizon)
    surrogate_loss = velocity_loss + omega * windowed_waypoint_loss

    loss = (quadratic_loss.detach() + surrogate_loss - surrogate_loss.detach())
```

**说明**：  
- 不使用截断时：直接返回 `quadratic_loss`，梯度为标准 $2Pe$。  
- 使用截断时：  
  - `surrogate_loss` 使用了 `detached_integral` 计算位置，因此其梯度对应窗口截断的代理梯度。  
  - `loss` 的前向数值：`quadratic_loss`（因为 `surrogate_loss - surrogate_loss.detach()` 前向抵消）。  
  - `loss` 的反向梯度：`∇ surrogate_loss`（因为 `quadratic_loss.detach()` 和 `surrogate_loss.detach()` 梯度为零）。  
- 实现了前向精确、反向截断的效果。

---

## 总结

- 整个损失函数围绕“归一化误差”构建，并利用积分矩阵 $M$ 将速度监督与位置监督统一为二次型 $P$ 范数。  
- $P$ 矩阵正定，保证理论最优解不变。  
- 截断梯度通过 `detached_integral` 和 `detach` 技巧实现，仅改变反向传播路径，不影响前向损失数值。